# YOLOv9-S Fire and Smoke Detection Training

Google Colab notebook used for dataset preparation, YOLOv9-S training, validation, and experiment output collection.


## 0. Runtime and GPU Check


In [ ]:

import os
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU。请先在 Colab 的运行时设置中选择 GPU。")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("显存 GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

!nvidia-smi


## 1. Google Drive Mount


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/fire_smoke_project")
DATA_ROOT = Path("/content/datasets/fire_smoke")
RUNS_ROOT = DRIVE_ROOT / "runs"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive project:", DRIVE_ROOT)
print("Local dataset:", DATA_ROOT)
print("Training outputs:", RUNS_ROOT)


## 2. Environment Setup


In [ ]:

!pip install -q -U ultralytics kagglehub pyyaml


In [ ]:

import ultralytics
import kagglehub
import yaml
import numpy as np
import cv2
import matplotlib.pyplot as plt

print("Ultralytics:", ultralytics.__version__)
print("KaggleHub:", getattr(kagglehub, "__version__", "unknown"))
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)


## 3. Kaggle Authentication


In [ ]:

import kagglehub

# Kaggle authentication may be requested on the first run.
# Public datasets may download without authentication.
kagglehub.login()


## 4. Dataset Download


In [ ]:

import shutil
from pathlib import Path
import kagglehub

DATASET_HANDLE = "sayedgamal99/smoke-fire-detection-yolo"

# Remove an incomplete download before retrying.
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

downloaded_path = kagglehub.dataset_download(
    DATASET_HANDLE,
    output_dir=str(DATA_ROOT),
)

print("下载完成：", downloaded_path)


## 5. Dataset Directory Inspection


In [ ]:

from pathlib import Path

def print_tree(root: Path, max_depth: int = 4, max_items: int = 250):
    root = root.resolve()
    count = 0
    for path in sorted(root.rglob("*")):
        try:
            depth = len(path.relative_to(root).parts)
        except ValueError:
            continue
        if depth > max_depth:
            continue
        prefix = "    " * (depth - 1)
        marker = "📁" if path.is_dir() else "📄"
        print(f"{prefix}{marker} {path.name}")
        count += 1
        if count >= max_items:
            print(f"... 已停止显示，当前共显示 {max_items} 项")
            break

print_tree(DATA_ROOT)


In [ ]:
print("DATA_ROOT =", DATA_ROOT)
print("路径是否存在 =", DATA_ROOT.exists())

all_items = list(DATA_ROOT.rglob("*"))
print("文件和文件夹总数 =", len(all_items))

for item in all_items[:30]:
    print(item)

print("downloaded_path =", downloaded_path)

In [ ]:
import shutil
from pathlib import Path

SOURCE_ROOT = Path(downloaded_path)
DATA_ROOT = Path("/content/datasets/fire_smoke")

print("数据集来源：", SOURCE_ROOT)
print("来源是否存在：", SOURCE_ROOT.exists())

DATA_ROOT.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    SOURCE_ROOT,
    DATA_ROOT,
    dirs_exist_ok=True
)

all_items = list(DATA_ROOT.rglob("*"))

print("复制完成")
print("DATA_ROOT =", DATA_ROOT)
print("文件和文件夹总数 =", len(all_items))

for item in all_items[:30]:
    print(item)

In [ ]:
yaml_files = sorted(
    list(DATA_ROOT.rglob("*.yaml"))
    + list(DATA_ROOT.rglob("*.yml"))
)

print("找到的 YAML 文件：")

for i, path in enumerate(yaml_files):
    print(i, path)

if not yaml_files:
    raise FileNotFoundError(
        "没有找到 data.yaml。请检查数据集是否确实是 YOLO 目标检测格式。"
    )

## 6. Dataset YAML Selection


In [ ]:

YAML_INDEX = 0
DATA_YAML = yaml_files[YAML_INDEX]

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("选中的 YAML：", DATA_YAML)
print(yaml.safe_dump(data_cfg, allow_unicode=True, sort_keys=False))


In [ ]:

required_keys = {"train", "val", "names"}
missing = required_keys - set(data_cfg.keys())

if missing:
    raise ValueError(f"data.yaml 缺少字段：{missing}")

print("类别 names:", data_cfg["names"])
print("train:", data_cfg["train"])
print("val:", data_cfg["val"])
print("test:", data_cfg.get("test", "未提供"))


## 7. Dataset Path Resolution


In [ ]:

from pathlib import Path

def resolve_yaml_base(data_yaml: Path, cfg: dict) -> Path:
    yaml_dir = data_yaml.parent
    declared_root = cfg.get("path")
    if declared_root is None:
        return yaml_dir

    declared_root = Path(str(declared_root))
    if declared_root.is_absolute() and declared_root.exists():
        return declared_root

    candidate = (yaml_dir / declared_root).resolve()
    if candidate.exists():
        return candidate

    # The source YAML may contain an obsolete absolute dataset path.
    # Fall back to the YAML directory when resolving relative paths.
    return yaml_dir.resolve()

def resolve_split_path(data_yaml: Path, cfg: dict, split: str):
    value = cfg.get(split)
    if value is None:
        return None

    base = resolve_yaml_base(data_yaml, cfg)

    if isinstance(value, list):
        return [resolve_one_path(base, data_yaml.parent, v) for v in value]

    return resolve_one_path(base, data_yaml.parent, value)

def resolve_one_path(base: Path, yaml_dir: Path, value):
    p = Path(str(value))
    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            base / p,
            yaml_dir / p,
            DATA_ROOT / p,
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return candidates[0].resolve()

for split in ["train", "val", "test"]:
    print(split, "=>", resolve_split_path(DATA_YAML, data_cfg, split))


In [ ]:

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_count(path):
    if path is None:
        return 0
    if isinstance(path, list):
        return sum(image_count(p) for p in path)
    path = Path(path)
    if path.is_file() and path.suffix.lower() == ".txt":
        return sum(1 for line in path.read_text().splitlines() if line.strip())
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTS)

for split in ["train", "val", "test"]:
    path = resolve_split_path(DATA_YAML, data_cfg, split)
    print(f"{split}: {path} | images={image_count(path)}")


### Optional: Generate a Colab-Specific Dataset YAML


In [ ]:

def find_image_dirs(root: Path):
    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        if p.name.lower() != "images":
            continue
        parent_name = p.parent.name.lower()
        if parent_name in {"train", "training", "val", "valid", "validation", "test", "testing"}:
            candidates.append(p.resolve())
    return sorted(candidates)

image_dirs = find_image_dirs(DATA_ROOT)
for p in image_dirs:
    print(p, "images=", image_count(p))


In [ ]:

# Use this cell only when the source data.yaml paths are invalid.
# Example:
# TRAIN_IMAGES = Path("/content/datasets/fire_smoke/train/images")
# VAL_IMAGES   = Path("/content/datasets/fire_smoke/valid/images")
# TEST_IMAGES  = Path("/content/datasets/fire_smoke/test/images")

TRAIN_IMAGES = None
VAL_IMAGES = None
TEST_IMAGES = None

if TRAIN_IMAGES is not None and VAL_IMAGES is not None:
    fixed_cfg = {
        "path": str(DATA_ROOT),
        "train": str(Path(TRAIN_IMAGES).resolve()),
        "val": str(Path(VAL_IMAGES).resolve()),
        "names": data_cfg["names"],
    }
    if TEST_IMAGES is not None:
        fixed_cfg["test"] = str(Path(TEST_IMAGES).resolve())

    FIXED_YAML = DATA_ROOT / "data_colab.yaml"
    with open(FIXED_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(fixed_cfg, f, allow_unicode=True, sort_keys=False)

    DATA_YAML = FIXED_YAML
    data_cfg = fixed_cfg
    print("已生成：", DATA_YAML)
    print(yaml.safe_dump(data_cfg, allow_unicode=True, sort_keys=False))
else:
    print("未生成新 YAML；继续使用原始 YAML：", DATA_YAML)


## 8. Image and Label Validation


In [ ]:

from collections import Counter
from pathlib import Path

def collect_images(split_path):
    if split_path is None:
        return []
    if isinstance(split_path, list):
        images = []
        for p in split_path:
            images.extend(collect_images(p))
        return images

    p = Path(split_path)
    if p.is_file() and p.suffix.lower() == ".txt":
        return [Path(line.strip()) for line in p.read_text().splitlines() if line.strip()]

    if not p.exists():
        return []

    return sorted(x for x in p.rglob("*") if x.suffix.lower() in IMAGE_EXTS)

def corresponding_label(image_path: Path):
    parts = list(image_path.parts)
    lower_parts = [x.lower() for x in parts]

    if "images" in lower_parts:
        idx = len(lower_parts) - 1 - lower_parts[::-1].index("images")
        parts[idx] = "labels"
        return Path(*parts).with_suffix(".txt")

    return image_path.with_suffix(".txt")

class_counts = Counter()
invalid_rows = []
missing_labels = []
split_summary = {}

for split in ["train", "val", "test"]:
    split_path = resolve_split_path(DATA_YAML, data_cfg, split)
    images = collect_images(split_path)
    if not images:
        continue

    labels_found = 0
    empty_labels = 0

    for image in images:
        label = corresponding_label(image)
        if not label.exists():
            missing_labels.append(str(image))
            continue

        labels_found += 1
        lines = [line.strip() for line in label.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            empty_labels += 1

        for line_number, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                invalid_rows.append((str(label), line_number, line, "列数不是5"))
                continue
            try:
                cls = int(float(parts[0]))
                coords = [float(v) for v in parts[1:]]
            except ValueError:
                invalid_rows.append((str(label), line_number, line, "无法转为数字"))
                continue

            if not all(0 <= v <= 1 for v in coords):
                invalid_rows.append((str(label), line_number, line, "坐标超出0~1"))
                continue

            class_counts[cls] += 1

    split_summary[split] = {
        "images": len(images),
        "labels_found": labels_found,
        "empty_labels": empty_labels,
    }

print("各划分统计：", split_summary)
print("各类别框数量：", dict(class_counts))
print("缺失标签图片数：", len(missing_labels))
print("无效标签行数：", len(invalid_rows))

if missing_labels:
    print("缺失标签示例：", missing_labels[:5])
if invalid_rows:
    print("无效标签示例：", invalid_rows[:5])


## 9. Annotation Visualization


In [ ]:

import random
import matplotlib.pyplot as plt
import cv2
import numpy as np

names_cfg = data_cfg["names"]
if isinstance(names_cfg, dict):
    class_names = {int(k): v for k, v in names_cfg.items()}
else:
    class_names = {i: name for i, name in enumerate(names_cfg)}

train_images = collect_images(resolve_split_path(DATA_YAML, data_cfg, "train"))
sample_images = random.sample(train_images, min(6, len(train_images)))

plt.figure(figsize=(16, 10))

for plot_index, image_path in enumerate(sample_images, 1):
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        continue
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    label_path = corresponding_label(image_path)
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            cls, xc, yc, bw, bh = map(float, line.split())
            cls = int(cls)

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
            cv2.putText(
                img,
                class_names.get(cls, str(cls)),
                (max(0, x1), max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

    plt.subplot(2, 3, plot_index)
    plt.imshow(img)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 10. Load Pretrained YOLOv9-S


In [ ]:

from ultralytics import YOLO

model = YOLO("yolov9s.pt")
model.info()


## 11. Five-Epoch Smoke Test


In [ ]:
from pathlib import Path
import yaml
import shutil

# Dataset location.
ROOT = Path("/content/datasets/fire_smoke")
ORIGINAL_YAML = ROOT / "data.yaml"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0

    return sum(
        1
        for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

# Locate image directories automatically.
image_dirs = []

for folder in ROOT.rglob("images"):
    if folder.is_dir():
        image_dirs.append(
            {
                "path": folder.resolve(),
                "split_name": folder.parent.name.lower(),
                "count": count_images(folder),
            }
        )

print("找到的图片目录：")

for item in image_dirs:
    print(
        item["split_name"],
        "| 图片数：", item["count"],
        "| 路径：", item["path"]
    )

# Common dataset split names.
split_aliases = {
    "train": {"train", "training"},
    "val": {"val", "valid", "validation"},
    "test": {"test", "testing"},
}

def choose_split(split: str):
    aliases = split_aliases[split]

    candidates = [
        item
        for item in image_dirs
        if item["split_name"] in aliases and item["count"] > 0
    ]

    if not candidates:
        return None

    # Select the candidate containing the largest image set.
    candidates.sort(key=lambda x: x["count"], reverse=True)
    return candidates[0]["path"]

TRAIN_IMAGES = choose_split("train")
VAL_IMAGES = choose_split("val")
TEST_IMAGES = choose_split("test")

print("\n自动选择的路径：")
print("TRAIN_IMAGES =", TRAIN_IMAGES)
print("VAL_IMAGES   =", VAL_IMAGES)
print("TEST_IMAGES  =", TEST_IMAGES)

if TRAIN_IMAGES is None:
    raise FileNotFoundError("没有找到训练集 images 文件夹。")

if VAL_IMAGES is None:
    raise FileNotFoundError("没有找到验证集 images 文件夹。")

# Read class names from the original YAML.
with open(ORIGINAL_YAML, "r", encoding="utf-8") as file:
    original_cfg = yaml.safe_load(file)

if "names" not in original_cfg:
    raise KeyError("原始 data.yaml 中没有 names 类别信息。")

# Generate a Colab-specific YAML using resolved paths.
fixed_cfg = {
    "train": str(TRAIN_IMAGES),
    "val": str(VAL_IMAGES),
    "names": original_cfg["names"],
}

if TEST_IMAGES is not None:
    fixed_cfg["test"] = str(TEST_IMAGES)

FIXED_YAML = ROOT / "data_colab.yaml"

with open(FIXED_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        fixed_cfg,
        file,
        allow_unicode=True,
        sort_keys=False
    )

# Update the configuration path used by later cells.
DATA_YAML = FIXED_YAML
data_cfg = fixed_cfg

print("\n新的 data.yaml 已生成：")
print(DATA_YAML)

print("\n新的 YAML 内容：")
print(
    yaml.safe_dump(
        data_cfg,
        allow_unicode=True,
        sort_keys=False
    )
)

# Final check.
assert Path(data_cfg["train"]).exists(), "训练集路径不存在"
assert Path(data_cfg["val"]).exists(), "验证集路径不存在"

print("路径检查通过，可以继续训练。")

In [ ]:
print("当前训练使用的 YAML：", DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
print("当前训练使用的 YAML：", DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
import shutil

FAILED_RUN = RUNS_ROOT / "yolov9s_test_5e"

shutil.rmtree(
    FAILED_RUN,
    ignore_errors=True
)

print("旧测试目录已清理：", FAILED_RUN)

In [ ]:

from ultralytics import YOLO

model = YOLO("yolov9s.pt")

test_results = model.train(
    data=str(DATA_YAML),
    epochs=5,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUNS_ROOT),
    name="yolov9s_test_5e",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    deterministic=True,
    amp=True,
    plots=True,
    save=True,
    save_period=1,
    cache=False,
    verbose=True,
)


## 12. Smoke-Test Outputs


In [ ]:

TEST_RUN_DIR = RUNS_ROOT / "yolov9s_test_5e"

for p in sorted(TEST_RUN_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(TEST_RUN_DIR))


In [ ]:

from IPython.display import Image, display

for filename in [
    "results.png",
    "confusion_matrix.png",
    "PR_curve.png",
    "F1_curve.png",
]:
    path = TEST_RUN_DIR / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path), width=900))


## 13. Validation of `best.pt`


In [ ]:

from ultralytics import YOLO

BEST_TEST_WEIGHT = TEST_RUN_DIR / "weights" / "best.pt"
assert BEST_TEST_WEIGHT.exists(), f"未找到 {BEST_TEST_WEIGHT}"

best_test_model = YOLO(str(BEST_TEST_WEIGHT))

eval_split = "test" if data_cfg.get("test") else "val"
metrics = best_test_model.val(
    data=str(DATA_YAML),
    split=eval_split,
    imgsz=640,
    batch=8,
    device=0,
    project=str(RUNS_ROOT),
    name=f"yolov9s_test_5e_eval_{eval_split}",
    exist_ok=True,
    plots=True,
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("各类别 mAP50-95:", metrics.box.maps)


## 14. Qualitative Prediction Check


In [ ]:

prediction_source = resolve_split_path(
    DATA_YAML,
    data_cfg,
    "test" if data_cfg.get("test") else "val"
)

prediction_images = collect_images(prediction_source)
selected = random.sample(prediction_images, min(10, len(prediction_images)))

pred_results = best_test_model.predict(
    source=[str(p) for p in selected],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(RUNS_ROOT),
    name="yolov9s_test_predictions",
    exist_ok=True,
)


In [ ]:

PRED_DIR = RUNS_ROOT / "yolov9s_test_predictions"
predicted_images = [
    p for p in PRED_DIR.rglob("*")
    if p.suffix.lower() in IMAGE_EXTS
]

plt.figure(figsize=(16, 12))
for i, path in enumerate(predicted_images[:9], 1):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    plt.subplot(3, 3, i)
    plt.imshow(img)
    plt.title(path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 15. Formal YOLOv9-S Training


In [ ]:
!pip install -q ultralytics==8.4.114 kagglehub pyyaml

from google.colab import drive
drive.mount("/content/drive")

import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

LAST_WEIGHT = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e/weights/last.pt"
)

DATA_ROOT = Path("/content/datasets/fire_smoke")
DATA_YAML = DATA_ROOT / "data_colab.yaml"

TRAIN_IMAGES = DATA_ROOT / "data/train/images"
VAL_IMAGES = DATA_ROOT / "data/val/images"
TEST_IMAGES = DATA_ROOT / "data/test/images"

print("断点 last.pt：", LAST_WEIGHT.exists())
print("data_colab.yaml：", DATA_YAML.exists())
print("训练集：", TRAIN_IMAGES.exists())
print("验证集：", VAL_IMAGES.exists())
print("测试集：", TEST_IMAGES.exists())

if LAST_WEIGHT.exists():
    print(
        "last.pt 大小：",
        round(LAST_WEIGHT.stat().st_size / 1024**2, 2),
        "MB"
    )

In [ ]:
!pip install -q ultralytics==8.4.114 kagglehub pyyaml

from pathlib import Path
import shutil
import yaml
import kagglehub

DATASET_HANDLE = "sayedgamal99/smoke-fire-detection-yolo"
DATA_ROOT = Path("/content/datasets/fire_smoke")

# 1. Download the public Kaggle dataset.
downloaded_path = Path(
    kagglehub.dataset_download(DATASET_HANDLE)
)

print("数据集下载位置：", downloaded_path)
print("下载位置存在：", downloaded_path.exists())

# 2. Prepare the temporary dataset directory.
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

shutil.copytree(
    downloaded_path,
    DATA_ROOT,
    dirs_exist_ok=True
)

print("数据已复制到：", DATA_ROOT)
print("文件和文件夹数量：", len(list(DATA_ROOT.rglob("*"))))

# 3. Resolve image paths.
TRAIN_IMAGES = DATA_ROOT / "data/train/images"
VAL_IMAGES = DATA_ROOT / "data/val/images"
TEST_IMAGES = DATA_ROOT / "data/test/images"

print("\n训练集存在：", TRAIN_IMAGES.exists())
print("验证集存在：", VAL_IMAGES.exists())
print("测试集存在：", TEST_IMAGES.exists())

if not TRAIN_IMAGES.exists():
    raise FileNotFoundError(f"找不到训练集：{TRAIN_IMAGES}")

if not VAL_IMAGES.exists():
    raise FileNotFoundError(f"找不到验证集：{VAL_IMAGES}")

# 4. Read the original class order from data.yaml.
yaml_candidates = [
    p for p in DATA_ROOT.rglob("*.yaml")
    if p.name != "data_colab.yaml"
]

if not yaml_candidates:
    yaml_candidates = [
        p for p in DATA_ROOT.rglob("*.yml")
        if p.name != "data_colab.yml"
    ]

if not yaml_candidates:
    raise FileNotFoundError("没有找到原始 data.yaml")

ORIGINAL_YAML = yaml_candidates[0]

with open(ORIGINAL_YAML, "r", encoding="utf-8") as f:
    original_cfg = yaml.safe_load(f)

print("\n原始 YAML：", ORIGINAL_YAML)
print("类别顺序：", original_cfg["names"])

# 5. Generate a Colab-specific YAML.
fixed_cfg = {
    "train": str(TRAIN_IMAGES.resolve()),
    "val": str(VAL_IMAGES.resolve()),
    "test": str(TEST_IMAGES.resolve()),
    "names": original_cfg["names"],
}

DATA_YAML = DATA_ROOT / "data_colab.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        fixed_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("\n已生成：", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import torch

LAST_WEIGHT = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e/weights/last.pt"
)

print("GPU可用：", torch.cuda.is_available())
print("last.pt：", LAST_WEIGHT.exists())
print("data_colab.yaml：", DATA_YAML.exists())
print("训练集：", TRAIN_IMAGES.exists())
print("验证集：", VAL_IMAGES.exists())
print("测试集：", TEST_IMAGES.exists())

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "ultralytics==8.4.114",
    "pyyaml",
    "kagglehub",
])

import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

LAST_WEIGHT = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e/weights/last.pt"
)

print("last.pt：", LAST_WEIGHT.exists())

if LAST_WEIGHT.exists():
    print(
        "文件大小：",
        round(LAST_WEIGHT.stat().st_size / 1024**2, 2),
        "MB"
    )

In [ ]:
from pathlib import Path
import shutil
import yaml
import kagglehub

DATASET_HANDLE = "sayedgamal99/smoke-fire-detection-yolo"
DATA_ROOT = Path("/content/datasets/fire_smoke")

# Download the public dataset.
downloaded_path = Path(
    kagglehub.dataset_download(DATASET_HANDLE)
)

print("下载位置：", downloaded_path)

# Restore the dataset to the expected working path.
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

shutil.copytree(
    downloaded_path,
    DATA_ROOT,
    dirs_exist_ok=True
)

TRAIN_IMAGES = DATA_ROOT / "data/train/images"
VAL_IMAGES = DATA_ROOT / "data/val/images"
TEST_IMAGES = DATA_ROOT / "data/test/images"

print("训练集：", TRAIN_IMAGES.exists())
print("验证集：", VAL_IMAGES.exists())
print("测试集：", TEST_IMAGES.exists())

if not TRAIN_IMAGES.exists():
    raise FileNotFoundError(f"找不到训练集：{TRAIN_IMAGES}")

if not VAL_IMAGES.exists():
    raise FileNotFoundError(f"找不到验证集：{VAL_IMAGES}")

# Preserve the original class order.
ORIGINAL_YAML = DATA_ROOT / "data.yaml"

with open(ORIGINAL_YAML, "r", encoding="utf-8") as f:
    original_cfg = yaml.safe_load(f)

# Generate a YAML for the current Colab runtime.
fixed_cfg = {
    "train": str(TRAIN_IMAGES.resolve()),
    "val": str(VAL_IMAGES.resolve()),
    "test": str(TEST_IMAGES.resolve()),
    "names": original_cfg["names"],
}

DATA_YAML = DATA_ROOT / "data_colab.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        fixed_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("\n生成的 YAML：", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
import torch

print("GPU：", torch.cuda.is_available())
print("last.pt：", LAST_WEIGHT.exists())
print("data.yaml：", DATA_YAML.exists())
print("训练集：", TRAIN_IMAGES.exists())
print("验证集：", VAL_IMAGES.exists())
print("测试集：", TEST_IMAGES.exists())

In [ ]:
import torch
from pathlib import Path
from datetime import datetime

LAST_WEIGHT = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e/weights/last.pt"
)

print("last.pt 存在：", LAST_WEIGHT.exists())
print(
    "最后修改时间：",
    datetime.fromtimestamp(LAST_WEIGHT.stat().st_mtime)
)
print(
    "文件大小：",
    round(LAST_WEIGHT.stat().st_size / 1024**2, 2),
    "MB"
)

checkpoint = torch.load(
    LAST_WEIGHT,
    map_location="cpu",
    weights_only=False
)

saved_epoch = checkpoint.get("epoch", None)
train_args = checkpoint.get("train_args", {})

print("\ncheckpoint 内部 epoch 编号：", saved_epoch)

if saved_epoch is not None:
    print("已经完整完成的 epoch 数：", saved_epoch + 1)

print("原计划总 epoch：", train_args.get("epochs"))
print("原始数据配置：", train_args.get("data"))
print("原始保存目录：", train_args.get("save_dir"))

In [ ]:
from ultralytics import YOLO

resume_model = YOLO(str(LAST_WEIGHT))

resume_results = resume_model.train(
    resume=True
)

In [ ]:

from ultralytics import YOLO

formal_model = YOLO("yolov9s.pt")

formal_results = formal_model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUNS_ROOT),
    name="yolov9s_formal_100e",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    deterministic=True,
    amp=True,
    patience=30,
    close_mosaic=10,
    plots=True,
    save=True,
    save_period=5,
    cache=False,
    verbose=True,
)


## 16. Resume Training from Checkpoint


In [ ]:

from ultralytics import YOLO
from pathlib import Path

LAST_WEIGHT = RUNS_ROOT / "yolov9s_formal_100e" / "weights" / "last.pt"

if LAST_WEIGHT.exists():
    resumed_model = YOLO(str(LAST_WEIGHT))
    resumed_model.train(resume=True)
else:
    print("目前还没有 last.pt：", LAST_WEIGHT)


## 17. Final Validation


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import json
import platform
import torch
import ultralytics

# YOLOv9 experiment directory.
RUN_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e"
)

BEST_WEIGHT = RUN_DIR / "weights" / "best.pt"
DATA_YAML = Path("/content/datasets/fire_smoke/data_colab.yaml")

print("best.pt:", BEST_WEIGHT.exists())
print("data.yaml:", DATA_YAML.exists())

assert BEST_WEIGHT.exists(), "没有找到 best.pt"
assert DATA_YAML.exists(), "没有找到 data_colab.yaml"

# Use best.pt for final evaluation.
model = YOLO(str(BEST_WEIGHT))

test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUN_DIR.parent),
    name="yolov9s_final_test",
    exist_ok=True,
    plots=True,
)

# Convert arrays safely.
def to_list(value):
    if value is None:
        return []
    if hasattr(value, "tolist"):
        return value.tolist()
    return list(value)

class_maps = to_list(test_metrics.box.maps)
class_precision = to_list(getattr(test_metrics.box, "p", None))
class_recall = to_list(getattr(test_metrics.box, "r", None))
class_map50 = to_list(getattr(test_metrics.box, "ap50", None))

per_class = {}

for index, map_value in enumerate(class_maps):
    class_name = model.names[index]

    class_result = {
        "mAP50_95": float(map_value)
    }

    if index < len(class_precision):
        class_result["precision"] = float(class_precision[index])

    if index < len(class_recall):
        class_result["recall"] = float(class_recall[index])

    if index < len(class_map50):
        class_result["mAP50"] = float(class_map50[index])

    per_class[class_name] = class_result

summary = {
    "model": "YOLOv9-S",
    "evaluation_split": "test",
    "training_epochs": 100,
    "imgsz": 640,
    "batch": 8,
    "seed": 42,
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "mAP50": float(test_metrics.box.map50),
    "mAP50_95": float(test_metrics.box.map),
    "weight_size_mb": round(
        BEST_WEIGHT.stat().st_size / 1024**2, 3
    ),
    "parameters": int(
        sum(parameter.numel() for parameter in model.model.parameters())
    ),
    "speed_ms_per_image": test_metrics.speed,
    "per_class": per_class,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
}

SUMMARY_PATH = RUN_DIR / "final_test_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\n结果已保存到：", SUMMARY_PATH)

## 18. Environment Record


In [ ]:
import subprocess
import sys
import json
import shutil
import platform
import torch
import ultralytics

environment = {
    "model": "YOLOv9-S",
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "ultralytics": ultralytics.__version__,
}

ENV_PATH = RUN_DIR / "environment.json"
FREEZE_PATH = RUN_DIR / "pip_freeze.txt"
YAML_COPY_PATH = RUN_DIR / "data_colab_used.yaml"

ENV_PATH.write_text(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

with open(FREEZE_PATH, "w", encoding="utf-8") as file:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=file,
        text=True,
        check=True,
    )

shutil.copy2(DATA_YAML, YAML_COPY_PATH)

print("环境信息已保存：", ENV_PATH)
print("依赖列表已保存：", FREEZE_PATH)
print("数据配置已保存：", YAML_COPY_PATH)

## Experiment Outputs
